In [1]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)


In [2]:
import pandas as pd
pd.set_option("display.max_columns", None)

despesas = pd.read_parquet("data/processed/despesas.parquet")
despesas['vr_despesa_contratada_mi'] = despesas['vr_despesa_contratada'] / 1_000_000
doacoes = despesas[despesas['ds_origem_despesa'] == 'Doações financeiras a outros candidatos/partidos'].copy()

In [3]:
doacoes.query("ds_cargo_fornecedor == 'DEPUTADO FEDERAL'").agg({"vr_despesa_contratada_mi": "sum"})

vr_despesa_contratada_mi    43.50985
dtype: float64

In [4]:
doacoes.groupby(
    ["ds_cargo_fornecedor"], as_index=False
).agg({"vr_despesa_contratada_mi": "sum"})

,ds_cargo_fornecedor,vr_despesa_contratada_mi
0,#NULO,36.877231
1,1º SUPLENTE,0.270000
2,DEPUTADO DISTRITAL,6.641388
3,DEPUTADO ESTADUAL,229.166244
4,DEPUTADO FEDERAL,43.509850
5,GOVERNADOR,18.381139
6,PRESIDENTE,0.071841
7,SENADOR,7.025646
8,VICE-GOVERNADOR,0.489355


In [5]:
doacoes_fed = doacoes.loc[
    (doacoes["ds_cargo"] == "DEPUTADO FEDERAL")
    & (doacoes["ds_cargo_fornecedor"] == "DEPUTADO FEDERAL")
].copy()

In [6]:
doacoes_fed['mesmo_partido'] = doacoes_fed['sg_partido'] == doacoes_fed['sg_partido_fornecedor']

In [8]:
doacoes_fed.columns

Index(['ano_eleicao', 'sg_uf', 'nr_candidato', 'nm_candidato', 'sg_partido',
       'ds_cargo', 'sg_uf_fornecedor', 'nr_candidato_fornecedor',
       'nm_fornecedor', 'sg_partido_fornecedor', 'ds_cargo_fornecedor',
       'ds_origem_despesa', 'dt_despesa', 'vr_despesa_contratada',
       'vr_despesa_contratada_mi', 'mesmo_partido'],
      dtype='object')

In [12]:
doacoes_fed.groupby("ano_eleicao").agg({"nr_candidato": "count", "vr_despesa_contratada_mi": "sum"})

,nr_candidato,vr_despesa_contratada_mi
ano_eleicao,,
2018,158,2.131087
2022,186,5.442200


In [ ]:
doacoes_fed.groupby("ano_eleicao")['vr_despesa_contratada_mi'].sum()

In [ ]:
doacoes_fed.groupby("mesmo_partido")['vr_despesa_contratada_mi'].sum()

In [ ]:
doacoes_fed['vr_despesa_contratada'].describe()

In [ ]:
receitas = pd.read_parquet("data/processed/receitas.parquet")
receitas['vr_receita_mi'] = receitas['vr_receita'] / 1_000_000

In [ ]:
receitas[receitas['ds_origem_receita'] == 'Recursos de outros candidatos'].groupby("ds_cargo").agg({"vr_receita_mi": "sum"})

## Reconciliação receitas × despesas: "Recursos de outros candidatos"

A parquet processada de receitas não tem `ds_especie_receita` (foi descartada no agrupamento do bronze).
Lemos o CSV bruto para separar **Estimado** (bens/serviços) de **Financeiro** (dinheiro vivo).

**Hipótese**: receitas financeiras ≈ despesas "Doações financeiras a outros candidatos/partidos";
a diferença de ~R$ 22 mi é inteiramente de receitas **Estimado**, sem contrapartida nessa categoria de despesa.

In [ ]:

RAW = "data/raw/finanças"
CARGO = "DEPUTADO FEDERAL"
ORIGEM_REC = "Recursos de outros candidatos"
ORIGEM_DESP = "Doações financeiras a outros candidatos/partidos"

# --- Receitas brutas (para ter ds_especie_receita) ---
_dtype = {"CD_ESFERA_PARTIDARIA_DOADOR": "str", "NR_DOCUMENTO_DOACAO": "str"}
rec_raw = pd.concat([
    pd.read_csv(f"{RAW}/receitas_candidatos_{ano}_BRASIL.csv", sep=";", encoding="latin1", dtype=_dtype)
    for ano in [2018, 2022]
])
rec_raw.columns = rec_raw.columns.str.lower()
rec_raw["vr_receita"] = rec_raw["vr_receita"].str.replace(",", ".").astype(float)
rec_raw["ds_cargo"] = rec_raw["ds_cargo"].str.upper()

rec_fed = rec_raw[
    (rec_raw["ds_cargo"] == CARGO) &
    (rec_raw["ds_origem_receita"] == ORIGEM_REC)
].copy()
rec_fed["vr_mi"] = rec_fed["vr_receita"] / 1_000_000
rec_fed["tipo"] = rec_fed["ds_especie_receita"].str.strip().apply(
    lambda x: "Estimado" if str(x).lower() == "estimado" else "Financeiro"
)

rec_por_ano = (
    rec_fed.groupby(["ano_eleicao", "tipo"])["vr_mi"]
    .sum()
    .unstack(fill_value=0)
    .assign(Total=lambda df: df.sum(axis=1))
    .round(2)
)

# --- Despesas: doações financeiras cujo destinatário é deputado federal ---
desp_fed = despesas[
    (despesas["ds_cargo_fornecedor"] == CARGO) &
    (despesas["ds_origem_despesa"] == ORIGEM_DESP)
].copy()

desp_por_ano = (
    desp_fed.groupby("ano_eleicao")["vr_despesa_contratada_mi"]
    .sum()
    .rename("Desp_Financeiro")
    .round(2)
)

# --- Tabela de reconciliação ---
reconciliacao = rec_por_ano.join(desp_por_ano)
reconciliacao["Diferença (Rec_Fin - Desp_Fin)"] = (
    reconciliacao.get("Financeiro", 0) - reconciliacao["Desp_Financeiro"]
).round(2)

print("Receitas 'Recursos de outros candidatos' (DEPUTADO FEDERAL) × Despesas 'Doações financeiras' → Deputado Federal\n")
print(reconciliacao.to_string())
print()
print("Totais:")
print(reconciliacao.sum().round(2))
